In [1]:
import cv2
from ultralytics import YOLO

model = YOLO("C:\\Users\\Cemal\\academic_projects\\runs\\detect\\train\\weights\\best.pt")
cap = cv2.VideoCapture("C:\\Users\\Cemal\\academic_projects\\Fish_Projects\\cemal_örnek_veri\\Scenario-1\\5\\GoPro12\\GX011530.MP4")

roi = None
counted_ids = set()  # Nesne ID'lerini saklamak için

while True:
    ret, frame = cap.read()
    if not ret:
        break

    key = cv2.waitKey(1) & 0xFF

    if key == ord('r'):
        temp = frame.copy()
        roi_box = cv2.selectROI("ROI Seç", temp, showCrosshair=False, fromCenter=False)
        if roi_box != (0, 0, 0, 0):
            roi = tuple(map(int, roi_box))
        cv2.destroyWindow("ROI Seç")
    
    if roi:
        x, y, w, h = roi
        
        # Tüm frame'de nesne takibi yap
        results = model.track(frame, persist=True)[0]
        boxes = results.boxes.xyxy.cpu().numpy()#type:ignore
        ids = results.boxes.id.cpu().numpy() if results.boxes.id is not None else None #type:ignore
        
        # ROI sınırları
        roi_x1, roi_y1, roi_x2, roi_y2 = x, y, x + w, y + h
        
        if ids is not None:
            for box, obj_id in zip(boxes, ids):
                x1, y1, x2, y2 = map(int, box)
                obj_id = int(obj_id)
                
                # Nesnenin merkezini hesapla
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2
                
                # Merkez ROI içinde mi kontrol et
                if roi_x1 <= cx <= roi_x2 and roi_y1 <= cy <= roi_y2:
                    if obj_id not in counted_ids:
                        counted_ids.add(obj_id)
                    
                    # Çerçeve üzerine ROI içindeki nesneleri işaretle
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(frame, f"ID: {obj_id}", (x1, y1-10), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)
        
        # ROI dikdörtgenini ve toplam sayımı çiz
        cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 255), 2)
        cv2.putText(frame, f"Toplam Balik: {len(counted_ids)}", (x, y-30), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

    cv2.imshow("Video", frame)

    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


0: 384x640 4 S_auratas, 118.5ms
Speed: 5.8ms preprocess, 118.5ms inference, 9.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 S_auratas, 66.6ms
Speed: 4.5ms preprocess, 66.6ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 S_auratas, 60.0ms
Speed: 2.9ms preprocess, 60.0ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 S_auratas, 60.3ms
Speed: 3.2ms preprocess, 60.3ms inference, 2.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 S_auratas, 62.8ms
Speed: 2.9ms preprocess, 62.8ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 S_auratas, 58.7ms
Speed: 3.0ms preprocess, 58.7ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 S_auratas, 58.9ms
Speed: 3.3ms preprocess, 58.9ms inference, 6.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 S_auratas, 60.0ms
Speed: 2.8ms preprocess, 60.0ms inference, 1.0ms postprocess pe